In [23]:
#Simple RNN problem from kaggle - IMDB Dataset of 50K Movie Reviews

#imports plus dataset
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('IMDB Dataset.csv')


In [24]:
#data preprocessing


import re 
def text_preprocessing(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^A-Za-z0-9\s]', '', text)
    text = re.sub(r'<.*?>', '', text)
    return text

df['review'] = df['review'].apply(text_preprocessing)


In [25]:
#import nltk
#nltk.download('punkt')
#nltk.download('punkt_tab')
#nltk.download('stopwords')

In [26]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

def remove_stopword(text):
    stop_words = stopwords.words('english')  # Specify 'english' for English stopwords
    temp_text = word_tokenize(text)

    for word in temp_text:
        if word in stop_words:
            text=text.replace(word,"")
    return text

df['review'] = df['review'].apply(remove_stopword)

In [27]:
from nltk.stem import PorterStemmer

def Stemming(text):
    ps = PorterStemmer()
    tokens = word_tokenize(text)
    stemmed_words = []

    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    
    return ' '.join(stemmed_words)

df['review'] = df['review'].apply(Stemming)

In [28]:
df["sentiment"].replace("positive", 0, inplace=True)
df["sentiment"].replace("negative", 1, inplace=True)

C:\Users\sabin\AppData\Local\Temp\ipykernel_8776\3454234782.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["sentiment"].replace("positive", 0, inplace=True)
C:\Users\sabin\AppData\Local\Temp\ipykernel_8776\3454234782.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For e

In [29]:
#tokenizare simpla
from collections import Counter

def tokenize(text):
    return text.split()

df['tokens'] = df['review'].apply(tokenize)

#build vocab

all_words = [word for tokens in df['tokens'] for word in tokens]
word_counts = Counter(all_words)
vocab = {
    word: i+2 for i, (word, _) in enumerate(word_counts.items())
}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

In [30]:
#text -> index

def encode(tokens):
    return [vocab.get(word,vocab['<UNK>']) for word in tokens]

df['encoded'] = df['tokens'].apply(encode)

In [31]:
#padding 

max_len = 100 #lungime maxima

def pad(seq):
    if len(seq) < max_len:
        return seq + [0] * (max_len - len(seq))
    return seq[:max_len]

df['padded'] = df['encoded'].apply(pad)

In [32]:
X = np.array(df['padded'].tolist())
y = np.array(df['sentiment'].tolist())

from sklearn.model_selection import train_test_split

X_tr,X_val,y_tr,y_val = train_test_split(X,y,test_size = 0.2, random_state = 42)


In [33]:
print(type(X_tr))
print(type(X_tr[0]))
print(X_tr[0][:10])

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
[  236  1239  4794  3273   263   936  6277 20043 10708  1811]


In [34]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

class TextDataset(Dataset):
    def __init__(self,X,y):
        self.X = torch.tensor(X, dtype = torch.long)
        self.y = torch.tensor(y, dtype = torch.float32)
    
    def __len__(self):
        return len(self.X)

    def __getitem__(self,idx):
        return self.X[idx], self.y[idx]

train_dts = TextDataset(X_tr,y_tr)
val_dts = TextDataset(X_val,y_val)

train_loader = DataLoader(train_dts, batch_size = 64, shuffle = True)
val_loader = DataLoader(val_dts, batch_size = 64)

In [46]:
#model RNN

class RNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        #self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
    
    def forward(self,x):
        x = self.embedding(x)
        #out, _ = self.rnn(x)
        out, (h_n, c_n) = self.rnn(x)
        out = out[:, -1, :]
        out = self.fc(out)

        return out

In [47]:
#setup

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = RNNModel(
    len(vocab),   # vocab_size
    100,          # embed_dim
    128           # hidden_dim
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

In [48]:
#training
from tqdm import tqdm

EPOCHS = 10 

for epoch in range(EPOCHS):
    model.train()

    for X_batch, y_batch in tqdm(train_loader):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch).squeeze()
        loss = criterion(outputs,y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1} / {EPOCHS}, Loss : {loss.item():.4f}')

100%|██████████| 625/625 [00:28<00:00, 22.05it/s]


Epoch 1 / 10, Loss : 0.6833


100%|██████████| 625/625 [00:29<00:00, 21.29it/s]


Epoch 2 / 10, Loss : 0.6337


100%|██████████| 625/625 [00:27<00:00, 22.78it/s]


...

Epoch 8 / 10, Loss : 0.0307


100%|██████████| 625/625 [00:27<00:00, 23.00it/s]


Epoch 9 / 10, Loss : 0.0515


100%|██████████| 625/625 [00:26<00:00, 23.20it/s]

Epoch 10 / 10, Loss : 0.0256


In [49]:
#evaluation 

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X_batch,y_batch in val_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        outputs= model(X_batch).squeeze()
        preds = (torch.sigmoid(outputs) > 0.5).float()

        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)


print('Accuracy: ', correct/total)

Accuracy:  0.8264


In [ ]:
#tr_df, val_df = train_test_split(df, test_size = 0.3, 
 #                                random_state = 42, stratify = df['sentiment'])

: 